# Stage 1 extraction noise + spectrum fidelity self-check

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [GitHub](https://github.com/oirlab/HISPEC_DRP).</span>

This notebook has two goals:

1. **2-D extract residual self-check** — controlled sheared-SP world with Stage-0 noise;
   pixel pulls ``(D−M)/rate_err`` should be near N(0,1).
2. **1-D spectrum fidelity vs ``build_l2``** — continuum-normalized comparison of
   box and sheared-SP extracts from the noiseless rate map against the ideal
   Lanczos-painted L2 (same template / RV). Gaps (LSF, separable model, box
   smear) are quantified for stellar-RV readiness.

> **Setup.** ``KOA_CALIBRATION_CACHE`` (optional Keck Observer credentials for remote sync).

In [1]:
import os

import hispecdrp

OUT = "/Users/ckilpatrick/stage1_extraction_noise_selfcheck"
os.makedirs(OUT, exist_ok=True)
KOA_CACHE = os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE")
os.environ.setdefault("KOA_CALIBRATION_CACHE", KOA_CACHE)
FIG_DIR = os.path.join(OUT, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print(f"hispecdrp {hispecdrp.__version__}")
print(f"OUT={OUT}")
print(f"KOA_CALIBRATION_CACHE={os.environ['KOA_CALIBRATION_CACHE']}")

hispecdrp 0.0.post925+ge8d36d9f6.d20260824
OUT=/Users/ckilpatrick/stage1_extraction_noise_selfcheck
KOA_CALIBRATION_CACHE=/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE


## 1. KOA caldb

Uses ``ensure_koa_calibrations``: seed the notebook cache if needed, try remote
sync, then stay offline for Stage 0 / simulation (``HISPEC_KOA_CONNECT_REMOTE=0``).

In [2]:
from hispecdrp.calibrations import ensure_koa_calibrations
from hispecdrp.datamodels.hispec_config import HISPECConfig

SPECTROGRAPH = "BSPEC"
FIBER = 2
EXTRACTION_APERTURE = 8.0

ORDER_OFFSET = HISPECConfig.echelle_orders[SPECTROGRAPH]["order_start"]
ZOOM_ORDERS = [ORDER_OFFSET, ORDER_OFFSET + 1]

# Optional: seed from another populated cache, e.g.
# KOA_SEED_SOURCES = ["/path/to/other_run/cals/KOA_CALIBRATION_CACHE"]
KOA_SEED_SOURCES = None

summary = ensure_koa_calibrations(
    SPECTROGRAPH,
    cache_dir=KOA_CACHE,
    seed_sources=KOA_SEED_SOURCES,
    ignore_missing_cal_types=["lfc_wavesol"],
)

os.environ["HISPEC_KOA_CONNECT_REMOTE"] = "0"

seed = summary.get("seed") or {}
if seed.get("seeded"):
    print(
        f"seeded KOA cache from {seed['source']}  "
        f"({seed['n_fits_before']} -> {seed['n_fits_after']} FITS)"
    )
elif seed.get("n_fits_before", 0):
    print(f"KOA cache already populated ({seed['n_fits_before']} FITS)")
print(f"remote_mode: {summary.get('remote_mode')}")
print(f"ORDER_OFFSET={ORDER_OFFSET}  ZOOM_ORDERS={ZOOM_ORDERS}")
print(f"{summary['spectrograph']}: {summary['n_records']} records")

[2026-08-24T20:51:16][hispecdrp.calibrations.koa_sync:405][INFO]: KOA cache already populated (17 FITS) at /Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE; skip seed.
[2026-08-24T20:51:16][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'
[2026-08-24T20:51:17][koa_middleware.keck_client:81][INFO]: Session validation: no observer ID found; re-login required.
[2026-08-24T20:51:17][koa_middleware.keck_client:51][INFO]: No valid login detected, logging in...
[2026-08-24T20:51:19][koa_middleware.keck_client:136][INFO]: User info request successful: observer ID=4187
[2026-08-24T20:51:19][koa_middleware.keck_client:56][INFO]: 

## 2. Simulate uint16 UTR (same recipe as Stage 0 self-check)

In [3]:
import numpy as np
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [FIBER]
N_READS = 10
SEED = 0
TEMPLATE_FILENAME = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
RV_KM_S = 0.0  # must match create_ramp_from_spectrum radial_velocity_km_s

ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename=TEMPLATE_FILENAME,
    mag_ab=3.05,
    filter_band="J",
    n_reads=N_READS,
    seed=SEED,
    poisson_noise=True,
    convert_to_uint16=True,
    clip_ramps=True,
    use_master_calibrations=True,
    ramp_filename="simulated_ramp_uint16.fits",
    overwrite_ramp=True,
    radial_velocity_km_s=RV_KM_S,
)

truth_rate = np.asarray(rate_map, dtype=np.float64)
print(f"ramp dtype: {ramp.data.dtype}")
print(f"truth rate map: {truth_rate.shape}")
print(f"template={TEMPLATE_FILENAME}  RV={RV_KM_S} km/s")

[2026-08-24T20:51:21][hispecdrp.simulations.simulate_up_the_ramp:1545][INFO]: Building default rate map (fibers=[2], spectrograph=BSPEC, output_dir=/Users/ckilpatrick/stage1_extraction_noise_selfcheck).
[2026-08-24T20:51:21][hispecdrp.primitives.utils.spectral_utils:552][INFO]: Normalized spectrum to J=3.05 (AB mag); scale=5.8159e+04 (was 14.96 mag)
[2026-08-24T20:51:21][hispecdrp.simulations.simulate_up_the_ramp:1604][INFO]: Densifying fiber=2 spectrum to R>=3e5 before Doppler (PCHIP, densify-only)
[2026-08-24T20:51:23][hispecdrp.simulations.simulate_rate_map:524][INFO]: Creating native rate map (spectrograph=BSPEC, method=prf_convolution).
[2026-08-24T20:51:53][hispecdrp.simulations.simulate_rate_map:602][INFO]: Saving rate map to /Users/ckilpatrick/stage1_extraction_noise_selfcheck/rate_map.fits
[2026-08-24T20:51:53][hispecdrp.simulations.simulate_up_the_ramp:1642][INFO]: Fetching master calibrations from calibration store (spectrograph=BSPEC).
[2026-08-24T20:51:53][hispecdrp.calibr

## 3. Stage 0

In [4]:
from hispecdrp.pipelines import Stage0Pipeline

l0 = Stage0Pipeline.call(ramp, output_dir=OUT)

[2026-08-24T20:52:12][hispecdrp.pipelines.base_pipeline:327][INFO]: Applying Stage0Pipeline with parameters:
run = True
save_result = True
output_model = input
output_file = None
output_dir = /Users/ckilpatrick/stage1_extraction_noise_selfcheck
suffix = None
max_cores = 1
save_preview = False
show_preview = False
[2026-08-24T20:52:12][hispecdrp.primitives.base_primitive:335][INFO]: Applying DQInitPrimitive with parameters:
run = True
save_result = None
output_model = input
output_file = None
output_dir = None
suffix = dqinit
max_cores = 1
save_preview = False
show_preview = False
dq = None
[2026-08-24T20:52:12][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/stage1_extraction_noise_selfcheck/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table

## 4. Controlled sheared-SP self-check

- Continuum ``P``: spectrally smoothed noiseless rate (lines removed).
- Shear: ``∂row/∂col`` from HDF PSF stamps.
- Project stellar ``truth`` → separable ``M = A f`` (documents the HDF model floor).
- Controlled noisy frame: ``M + (L0.rate - truth)`` — real Stage-0 noise on a
  scene the extract model can fit exactly.
- Extract with banded LS (``sheared_sp``); rebuild with the same ``A``.

Full 2-D maps are kept only for ``ZOOM_ORDERS``.

In [ ]:
from hispecdrp.simulations.e2e_pipeline import (
    create_trace_shear_from_sim_hdf,
    create_tracepos_model_from_sim_hdf,
)
from hispecdrp.simulations.plotting import (
    collect_fiber_extraction_pulls,
    order_zoom_slice,
    plot_extraction_pull_histogram,
    plot_order_extraction_zoom_triptych,
    project_rate_onto_separable_model,
)

tracepos_model = create_tracepos_model_from_sim_hdf(SPECTROGRAPH)
tracepos = np.asarray(getattr(tracepos_model, f"pos_fib{FIBER}"), dtype=np.float64)
shear = create_trace_shear_from_sim_hdf(SPECTROGRAPH, fiber=FIBER)
print(
    f"shear shape={shear.shape}  "
    f"nanmedian|s|={np.nanmedian(np.abs(shear)):.4f} pix/pix"
)

# HDF stellar deposit vs separable Af (model floor; cannot be zero with Horne/SP-lite).
proj0 = project_rate_onto_separable_model(
    truth_rate,
    truth_rate,
    tracepos[ZOOM_ORDERS[0] - ORDER_OFFSET],
    extraction_aperture=EXTRACTION_APERTURE,
    shear=shear[ZOOM_ORDERS[0] - ORDER_OFFSET],
    continuum_smooth_pix=81,
)
print(
    f"HDF model floor (order {ZOOM_ORDERS[0]}): "
    f"rms(truth-Af)/rms(truth)={proj0['floor_frac']:.4f}"
)

# Controlled scene: separable Af + real Stage-0 noise (L0 - truth).
noise = np.asarray(l0.rate, dtype=np.float64) - truth_rate
model_sep = np.zeros_like(truth_rate)
for od in ZOOM_ORDERS:
    row = int(od) - ORDER_OFFSET
    proj = project_rate_onto_separable_model(
        truth_rate,
        truth_rate,
        tracepos[row],
        extraction_aperture=EXTRACTION_APERTURE,
        shear=shear[row],
        continuum_smooth_pix=81,
    )
    model_sep[proj["aperture_mask"]] = proj["model"][proj["aperture_mask"]]

rate_ctrl = model_sep + noise
extraction_results = collect_fiber_extraction_pulls(
    rate_ctrl,
    np.asarray(l0.rate_err),
    np.asarray(l0.dq),
    np.asarray(l0.rate_var_rnoise),
    np.asarray(l0.rate_var_poisson),
    truth_rate,
    tracepos,
    spectrograph=SPECTROGRAPH,
    orders=ZOOM_ORDERS,
    order_offset=ORDER_OFFSET,
    extraction_aperture=EXTRACTION_APERTURE,
    store_maps_orders=ZOOM_ORDERS,
    shear=shear,
    continuum_smooth_pix=81,
)
print(
    f"controlled: n_orders={extraction_results['n_orders']}  "
    f"mean|shear|={extraction_results['mean_abs_shear']:.4f}  "
    f"pixel pulls n={extraction_results['n_pix_pixel']}  "
    f"mu={np.mean(extraction_results['pulls_pixel']):.3f}  "
    f"sigma={np.std(extraction_results['pulls_pixel']):.3f}"
)

## 5. Zoomed 2-D maps (two orders)

Controlled rate → profile-subtracted residual → ``(D-M)/rate_err``.
Expect unstructured noise near N(0,1), not edge stripes.

In [ ]:
per_order = extraction_results["per_order"]

for od in ZOOM_ORDERS:
    row = int(od) - ORDER_OFFSET
    res = per_order[od]
    rslice, cslice = order_zoom_slice(tracepos[row])
    plot_order_extraction_zoom_triptych(
        rate_ctrl,
        res["residual"],
        res["pulls_pixel"],
        res["good_mask_pixel"],
        row_slice=rslice,
        col_slice=cslice,
        title=f"Order {od} controlled sheared-SP (aperture={EXTRACTION_APERTURE:g})",
        pull_label=r"$(D - M) / {\rm rate\_err}$",
        savefig=os.path.join(FIG_DIR, f"zoom_order{od}_controlled.png"),
    )

## 6. Pull histograms (N(0, 1) validation)

On the controlled scene, pixel pulls ``(D - M) / rate_err`` should match Stage 0:
Gaussian near N(0, 1). Spec×profile pulls are secondary (wing-sensitive).

In [ ]:
plot_extraction_pull_histogram(
    extraction_results["pulls_pixel"],
    title="Controlled sheared-SP pulls / rate_err",
    xlabel=r"$(D - M) / {\rm rate\_err}$",
    savefig=os.path.join(FIG_DIR, "extraction_pull_pixel_controlled.png"),
)
plot_extraction_pull_histogram(
    extraction_results["pulls_spec"],
    title="Controlled sheared-SP pulls / (spec_err * profile)",
    xlabel=r"$(D - M) / (\sigma_{\rm spec}\,P)$",
    savefig=os.path.join(FIG_DIR, "extraction_pull_spec_profile_controlled.png"),
)

## 7. Spectrum fidelity vs ``build_l2``

Ideal reference: :func:`~hispecdrp.simulations.simulate_spectra.build_l2_lsf_matched` paints the same
template onto the instrument WAVE grid (no detector PSF).

From the **noiseless** ``truth_rate`` map we extract:

- **box** — current E2E default geometry
- **sheared_sp** — continuum ``P`` + HDF iso-λ shear, banded LS

Both are continuum-normalized (median envelope) before comparison so blaze /
throughput shape drops out. Metrics: continuum-norm RMS (%), correlation,
median absorption-depth ratio (test / ideal).


In [ ]:
from hispecdrp.simulations.simulate_spectra import build_l2_lsf_matched
from hispecdrp.simulations.e2e_pipeline import extract_spectrum_from_rate_map
from hispecdrp.simulations.plotting import (
    compare_continuum_normalized_spectra,
    extract_sheared_sp_from_rate_map,
    extract_spectroperfectionism_from_rate_map,
    plot_spectrum_fidelity_overlay,
    summarize_fidelity_gaps,
)

# LSF-matched ideal L2 for like-to-like depth_ratio (instrument-apparent spectrum).
l2_ideal = build_l2_lsf_matched(
    RV_KM_S,
    spectrograph=SPECTROGRAPH,
    template=TEMPLATE_FILENAME,
    fiber=FIBER,
)
wave_ideal = np.asarray(getattr(l2_ideal, f"wave_fib{FIBER}"), dtype=np.float64)
flux_ideal = np.asarray(getattr(l2_ideal, f"spec_fib{FIBER}"), dtype=np.float64)
print(f"build_l2_lsf_matched shape={flux_ideal.shape}")

# Orders for extract-vs-ideal metrics. Use ZOOM_ORDERS for a fast interactive run;
# set FIDELITY_ORDERS = None and pass orders=None below for all BSPEC orders.
FIDELITY_ORDERS = list(ZOOM_ORDERS)

box = extract_spectrum_from_rate_map(
    truth_rate,
    spectrograph=SPECTROGRAPH,
    fiber=FIBER,
    extraction_aperture=EXTRACTION_APERTURE,
    orders=FIDELITY_ORDERS,
)
print(f"box extract: orders={box['orders']}")

print("sheared-SP extract...")
sp = extract_sheared_sp_from_rate_map(
    truth_rate,
    spectrograph=SPECTROGRAPH,
    fiber=FIBER,
    extraction_aperture=EXTRACTION_APERTURE,
    orders=FIDELITY_ORDERS,
    continuum_smooth_pix=81,
    shear=shear,
)
print(f"sheared_sp extract: orders={sp['orders']}")

print("spectroperfectionism extract (may be slow)...")
sp_full = extract_spectroperfectionism_from_rate_map(
    truth_rate,
    spectrograph=SPECTROGRAPH,
    fiber=FIBER,
    orders=FIDELITY_ORDERS,
)
print(f"SP extract: orders={sp_full['orders']}")

# Slice ideal L2 to the same echelle orders (row = order - ORDER_OFFSET).
ideal_rows = [int(od) - ORDER_OFFSET for od in FIDELITY_ORDERS]
wave_ref = wave_ideal[ideal_rows]
flux_ref = flux_ideal[ideal_rows]

fidelity_rows = []
for label, extr in [("box_from_rate_map", box), ("sheared_sp_from_rate_map", sp), ("spectroperfectionism", sp_full)]:
    row = compare_continuum_normalized_spectra(
        extr["wavelength"],
        extr["flux"],
        wave_ref,
        flux_ref,
        label=label,
    )
    fidelity_rows.append(row)
    print(
        f"{label}: rms={row['norm_rms_pct']:.3f}%  "
        f"corr={row['corr']:.4f}  depth_ratio={row['depth_median_ratio']:.3f}"
    )

print()
print(summarize_fidelity_gaps(fidelity_rows))


## 8. Per-order overlay (continuum-normalized)

Black = ``build_l2``. Colored = extracts. Bottom = test − ideal.
Line cores that are systematically shallower than ideal are the LSF / extract gap.

In [ ]:
box_cmp = next(r for r in fidelity_rows if r["label"] == "box_from_rate_map")
sp_cmp = next(r for r in fidelity_rows if r["label"] == "sheared_sp_from_rate_map")

for j, od in enumerate(FIDELITY_ORDERS):
    plot_spectrum_fidelity_overlay(
        wave_ref,
        box_cmp["ref_norm"],
        {
            "box": box_cmp["test_norm"],
            "sheared_sp": sp_cmp["test_norm"],
        },
        order_row=j,
        order_number=od,
        title=f"Order {od}: extract vs build_l2 (continuum-normalized)",
        savefig=os.path.join(FIG_DIR, f"fidelity_order{od}_vs_build_l2.png"),
    )

## 9. How to close the remaining gap (for cm/s RV)

| Gap | Symptom | Next step |
|---|---|---|
| Detector LSF / PSF | ``depth_ratio`` ≪ 1 vs **native** ``build_l2`` | Prefer ``build_l2_lsf_matched`` / ``*.lsf.ccfopt.delta.mas``; or ``method=spectroperfectionism`` |
| Separable extract | ~1% 2-D floor; residual line-shape error | Banded SP with true 2-D PSF kernels (not continuum ``P`` alone) |
| Box extract | Worse depth / RMS than sheared-SP | Prefer sheared-SP (or SP) for science extract; keep box for flats |
| WAVE / blaze | RV zero-point / continuum | Packaged line-ZP WAVE + fiberflat blaze on L2 (E2E path) |
| Mask mismatch | Aggregate RV OK, per-line MAD large | Build ccfopt mask from E2E blaze L2, not ideal-only |

This notebook isolates **extraction + LSF** relative to ideal paint. End-to-end RV
still needs Stage-2 blaze + barycorr + mask (see ``basic_usage`` / ``rv_recovery_sim``).
